# Session 11 — Advanced DES Practice: Inventory & Production Line Simulation

**Course:** Decision Modelling — Simulation Part
**Session:** 11 of 12 (simulation portion) — capstone practice session

## Learning objectives

By the end of this session you should be able to:

- Build and analyze a realistic, multi-component OSCM simulation model
- Identify bottlenecks in a multi-stage system
- Evaluate the performance of an inventory/production policy using the full analytical toolkit from this course

## Format note: flipped classroom

- **Pre-class (screencast + this notebook's Part A):** two new, focused SimPy mechanics — `simpy.Container` (modeling a bulk quantity like inventory level) and **finite-capacity buffers** (which create blocking and starvation). Watch the screencast and run Part A yourself before class.
- **In-class (Part B):** we assemble these into one integrated production/inventory system, and spend class time on bottleneck identification and policy evaluation — the genuinely OSCM-flavored analytical work this capstone is about.

## Where this session fits

- **This is the capstone practice session.** It combines multi-stage flow (Session 8), resource-based stations (Sessions 6–7), and now inventory + blocking/starvation — nearly everything from Sessions 6–10 in one model.
- **Looking ahead:** Session 12 is verification & validation only — you'll be asked to critically assess a model like the one you build today, not build something new.


## Further reading (supplementary, not required)

- A paper or case study on (s,S) inventory policies and on bottleneck identification via blocking/starvation analysis in production lines — check the course reading list for the current reference.


---
# Part A (pre-class): two new SimPy mechanics

Watch the accompanying screencast for a walkthrough of the code below, then run it yourself before class.

## A.1 `simpy.Container`: modeling a bulk quantity

Unlike `Resource` (discrete units of capacity, like servers) or `Store` (discrete individual items), `simpy.Container` models a **continuous or bulk quantity** — exactly what an inventory level is. `container.get(amount)` withdraws a quantity (blocking if insufficient is available... though below we'll check `level` first to model lost sales instead of waiting), and `container.put(amount)` adds a quantity.

Let's build a standalone **(s, S) inventory policy**: when the inventory level drops to or below the reorder point `s`, place a replenishment order that brings it back up to `S` after a lead time.


In [1]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

def check_reorder(env, container, policy_state, log):
    if container.level <= policy_state['s'] and not policy_state['order_outstanding']:
        order_qty = policy_state['S'] - container.level
        policy_state['order_outstanding'] = True
        env.process(place_order(env, container, order_qty, policy_state, log))

def place_order(env, container, order_qty, policy_state, log):
    yield env.timeout(policy_state['lead_time'])
    yield container.put(order_qty)
    policy_state['order_outstanding'] = False
    log.append({'time': env.now, 'event': 'replenishment_arrived', 'qty': order_qty, 'level_after': container.level})

def demand_consumer(env, container, mean_interarrival, rng, policy_state, log):
    while True:
        yield env.timeout(rng.exponential(mean_interarrival))
        if container.level >= 1:
            yield container.get(1)
            log.append({'time': env.now, 'event': 'demand_met', 'level_after': container.level})
        else:
            log.append({'time': env.now, 'event': 'stockout', 'level_after': container.level})
        check_reorder(env, container, policy_state, log)


In [2]:
rng = np.random.default_rng(seed=1)
env = simpy.Environment()

policy_state = {'s': 20, 'S': 100, 'lead_time': 8.0, 'order_outstanding': False}
container = simpy.Container(env, capacity=200, init=policy_state['S'])
log = []

env.process(demand_consumer(env, container, mean_interarrival=1.0, rng=rng, policy_state=policy_state, log=log))
env.run(until=500)

log_df = pd.DataFrame(log)
stockout_rate = (log_df['event'] == 'stockout').mean()
print(f"Stockout rate: {stockout_rate:.3%}")
print(log_df['event'].value_counts())


Stockout rate: 0.000%
event
demand_met               511
replenishment_arrived      6
Name: count, dtype: int64


## A.2 Finite buffers: blocking and starvation

A `simpy.Store` with a fixed `capacity` behaves exactly like an in-between buffer of limited size. If an upstream station tries to `put()` into a **full** buffer, it must wait — this is **blocking**. If a downstream station tries to `get()` from an **empty** buffer, it must wait — this is **starvation**. Both happen automatically; we just need to measure how long each wait lasts.


In [3]:
def station_upstream(env, buffer, mean_process_time, rng, log):
    while True:
        yield env.timeout(rng.exponential(mean_process_time))
        t0 = env.now
        yield buffer.put('item')
        blocked_time = env.now - t0
        if blocked_time > 1e-9:
            log.append({'time': env.now, 'event': 'blocked', 'duration': blocked_time})

def station_downstream(env, buffer, mean_process_time, rng, log):
    while True:
        t0 = env.now
        yield buffer.get()
        starved_time = env.now - t0
        if starved_time > 1e-9:
            log.append({'time': env.now, 'event': 'starved', 'duration': starved_time})
        yield env.timeout(rng.exponential(mean_process_time))

rng2 = np.random.default_rng(seed=2)
env2 = simpy.Environment()
buffer = simpy.Store(env2, capacity=3)
buffer_log = []

env2.process(station_upstream(env2, buffer, mean_process_time=1.0, rng=rng2, log=buffer_log))
env2.process(station_downstream(env2, buffer, mean_process_time=1.5, rng=rng2, log=buffer_log))
env2.run(until=1000)

buffer_log_df = pd.DataFrame(buffer_log)
print(buffer_log_df['event'].value_counts())
print(f"\nTotal time blocked (upstream station): {buffer_log_df[buffer_log_df.event=='blocked']['duration'].sum():.1f}")
print(f"Total time starved (downstream station): {buffer_log_df[buffer_log_df.event=='starved']['duration'].sum():.1f}")


event
blocked    255
starved     40
Name: count, dtype: int64

Total time blocked (upstream station): 353.0
Total time starved (downstream station): 36.6


Since the downstream station is *slower* (mean 1.5) than the upstream station (mean 1.0), you should see substantially more **blocking** than **starvation** — the upstream station keeps filling the buffer faster than the downstream station can drain it, so it frequently has nowhere to put its output. This is your first hint at **bottleneck identification**: heavy blocking upstream of a station is a signal that *that station* is the constraint. Part A is done here — the rest of this notebook (Part B) is for class.


---
# Part B (in-class): an integrated production & inventory capstone

## The system

We now combine everything into one model:

```
[Raw material Container, (s,S) policy]
        |
   [Station 1] --> [finite buffer] --> [Station 2]
                                             |
                                  [Finished goods Container]
                                             |
                                    [Customer demand -- lost sales if stocked out]
```

- **Station 1** consumes 1 unit of raw material per job, processes it, and passes it to the intermediate buffer
- **Station 2** pulls from the buffer, processes it, and adds 1 unit to finished-goods inventory
- **Customer demand** arrives randomly and is met from finished-goods stock if available; if not, it's a **lost sale** (no backorder)
- Raw material follows an **(s, S)** replenishment policy, exactly as in Part A


In [4]:
def station1(env, raw_material, buffer, mean_process_time, rng, policy_state, reorder_log, metrics):
    while True:
        yield raw_material.get(1)
        check_reorder(env, raw_material, policy_state, reorder_log)
        yield env.timeout(rng.exponential(mean_process_time))
        metrics['station1_busy_time'] += 0  # busy time already counted via the timeout above implicitly
        t0 = env.now
        yield buffer.put('item')
        blocked = env.now - t0
        metrics['station1_blocked_time'] += blocked

def station2(env, buffer, finished_goods, mean_process_time, rng, metrics):
    while True:
        t0 = env.now
        yield buffer.get()
        starved = env.now - t0
        metrics['station2_starved_time'] += starved
        yield env.timeout(rng.exponential(mean_process_time))
        yield finished_goods.put(1)

def demand_process(env, finished_goods, mean_interarrival, rng, metrics):
    while True:
        yield env.timeout(rng.exponential(mean_interarrival))
        metrics['total_demand'] += 1
        if finished_goods.level >= 1:
            yield finished_goods.get(1)
            metrics['demand_met'] += 1
        else:
            metrics['stockouts'] += 1


In [5]:
def run_production_system(run_length, buffer_capacity, s, S, lead_time,
                            mean_process_time_1, mean_process_time_2, mean_demand_interarrival, seed):
    rng = np.random.default_rng(seed)
    env = simpy.Environment()

    raw_material = simpy.Container(env, capacity=500, init=S)
    buffer = simpy.Store(env, capacity=buffer_capacity)
    finished_goods = simpy.Container(env, capacity=10000, init=0)

    policy_state = {'s': s, 'S': S, 'lead_time': lead_time, 'order_outstanding': False}
    reorder_log = []
    metrics = {'station1_busy_time': 0.0, 'station1_blocked_time': 0.0,
               'station2_starved_time': 0.0, 'total_demand': 0, 'demand_met': 0, 'stockouts': 0}

    env.process(station1(env, raw_material, buffer, mean_process_time_1, rng, policy_state, reorder_log, metrics))
    env.process(station2(env, buffer, finished_goods, mean_process_time_2, rng, metrics))
    env.process(demand_process(env, finished_goods, mean_demand_interarrival, rng, metrics))

    env.run(until=run_length)
    return metrics

run_length = 2000.0
metrics = run_production_system(
    run_length=run_length, buffer_capacity=5, s=20, S=100, lead_time=10.0,
    mean_process_time_1=1.5, mean_process_time_2=1.8, mean_demand_interarrival=2.0, seed=1
)

service_level = metrics['demand_met'] / metrics['total_demand']
station1_blocked_frac = metrics['station1_blocked_time'] / run_length
station2_starved_frac = metrics['station2_starved_time'] / run_length

print(f"Service level (demand met / total demand): {service_level:.3%}")
print(f"Station 1 -- fraction of time blocked:     {station1_blocked_frac:.3%}")
print(f"Station 2 -- fraction of time starved:      {station2_starved_frac:.3%}")
print(f"Total demand: {metrics['total_demand']}, stockouts: {metrics['stockouts']}")


Service level (demand met / total demand): 99.192%
Station 1 -- fraction of time blocked:     23.631%
Station 2 -- fraction of time starved:      5.233%
Total demand: 990, stockouts: 8


## 1. Bottleneck identification

Station 1's mean process time (1.5) is faster than Station 2's (1.8) — so we'd expect Station 1 to frequently find the buffer full (it produces faster than Station 2 can consume), and Station 2 to rarely starve (Station 1 keeps the buffer supplied). The blocking/starvation fractions above should confirm this. **The station with the high blocking rate upstream of it is the bottleneck** — here, that's Station 2. This is a genuinely useful, general diagnostic: in a real production line, persistent blocking at an upstream station is exactly how you'd spot a downstream capacity constraint without needing to instrument every station's utilization directly.


## 2. Evaluating policy performance with replications

A single run isn't enough to trust these numbers — bring back Session 8's replication approach to get a proper confidence interval on service level.


In [6]:
def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = data.mean()
    sem = data.std(ddof=1) / np.sqrt(n)
    half_width = stats.t.ppf((1 + confidence) / 2, df=n - 1) * sem
    return mean, half_width

n_reps = 20
service_levels = []
for i in range(n_reps):
    m = run_production_system(run_length=2000.0, buffer_capacity=5, s=20, S=100, lead_time=10.0,
                                mean_process_time_1=1.5, mean_process_time_2=1.8,
                                mean_demand_interarrival=2.0, seed=100 + i)
    service_levels.append(m['demand_met'] / m['total_demand'])

service_levels = np.array(service_levels)
mean_sl, hw_sl = confidence_interval(service_levels)
print(f"Service level across {n_reps} replications: {mean_sl:.3%}  (95% CI half-width: {hw_sl:.3%})")


Service level across 20 replications: 98.333%  (95% CI half-width: 0.630%)


## Exercise (guided): does buffer size fix the bottleneck?

Management wonders whether simply **increasing the intermediate buffer size** would improve service level, without touching Station 2's actual processing speed.

Your task:

1. Run the replication-based service-level analysis (as in Section 2) for `buffer_capacity` in `[2, 5, 10, 20]`, keeping all other parameters fixed
2. Present mean service level and its CI for each buffer size
3. Also record the mean Station 1 blocked-time fraction for each buffer size
4. State, in a sentence or two, whether increasing buffer size is an effective fix for this bottleneck, and why (or why not) — connect this back to what "bottleneck" actually means


In [7]:
# EXERCISE
buffer_sizes = [2, 5, 10, 20]
n_reps = 20

# TODO: for each buffer_capacity, run n_reps replications, computing service_level and
#       station1_blocked_time fraction for each; store mean + CI half-width per buffer size

# TODO: present results as a small table

# TODO: write a one-to-two sentence conclusion


<details>
<summary>Solution (click to expand)</summary>

```python
# SOLUTION
rows = []
for bc in buffer_sizes:
    sl_list = []
    blocked_list = []
    for i in range(n_reps):
        m = run_production_system(run_length=2000.0, buffer_capacity=bc, s=20, S=100, lead_time=10.0,
                                    mean_process_time_1=1.5, mean_process_time_2=1.8,
                                    mean_demand_interarrival=2.0, seed=200 + i)
        sl_list.append(m['demand_met'] / m['total_demand'])
        blocked_list.append(m['station1_blocked_time'] / 2000.0)

    sl_arr = np.array(sl_list)
    blocked_arr = np.array(blocked_list)
    mean_sl, hw_sl = confidence_interval(sl_arr)
    mean_blocked, hw_blocked = confidence_interval(blocked_arr)
    rows.append({'buffer_capacity': bc, 'mean_service_level': mean_sl, 'sl_half_width': hw_sl,
                 'mean_station1_blocked_frac': mean_blocked, 'blocked_half_width': hw_blocked})

buffer_df = pd.DataFrame(rows)
print(buffer_df.round(4))

print("\nConclusion: increasing buffer size reduces how often Station 1 is blocked (it has "
      "more room to deposit output), and modestly improves service level up to a point -- but "
      "it cannot fully compensate for Station 2 simply being the slower station on average. "
      "The buffer smooths out short-term variability, but the long-run finished-goods "
      "production rate is still capped by Station 2's processing rate; a persistently "
      "high blocking fraction even at large buffer sizes would be the signal that the real "
      "fix is Station 2's capacity, not the buffer.")
```
</details>


### Discussion questions

1. If you increased the buffer size to a very large number and blocking at Station 1 nearly disappeared, would that mean Station 2 is no longer the bottleneck? What would Little's Law (Session 5) say must still be true about the relationship between throughput and Station 2's capacity?
2. This model assumed lost sales (no backorders) for unmet demand. How do you think the (s, S) raw-material policy parameters would need to change if finished-goods demand could instead be backordered?
3. We used a single set of `(s, S, lead_time)` values for raw material throughout. What tool from Sessions 5 and 10 would you reach for to systematically search over `(s, S)` combinations for the best service level at the lowest inventory cost? (This is exactly where the integration/simulation-based-optimization sessions pick up.)


## Wrap-up

Today (Part A, pre-class) we:

- Introduced `simpy.Container` for modeling bulk inventory quantities, and built a standalone (s, S) replenishment policy
- Introduced finite-capacity buffers and observed blocking and starvation directly

Today (Part B, in-class) we:

- Assembled a full production/inventory system: (s,S) raw material -> Station 1 -> finite buffer -> Station 2 -> finished goods -> demand
- Used blocking/starvation patterns to identify the system's bottleneck without needing to instrument every station's utilization separately
- Applied Session 8's replication methodology to get a trustworthy confidence interval on service level
- Investigated whether buffer size alone can compensate for a slower downstream station — and connected the answer back to Little's Law

**Next session:** Verification & Validation — the final session of the simulation part. We step back and critically assess models like this one: are we confident it's correctly implemented (verification), and does it faithfully represent the real system it's meant to represent (validation)?
